## Analyze batter's decisions on pitches

This notebook builds a compact player-game model that summarizes recent pitch behavior and asks whether those patterns help explain offensive production.

The goal is:
- start with the raw Statcast pitch log
- convert pitch-level signals into player-game features
- create a custom per-plate-appearance offensive value target
- train a rolling-history model and compare it to a simple prior-performance baseline

The output is a predictive model for a batter's recent performance profile.


In [8]:
from pathlib import Path
import pandas as pd
import numpy as np

CURRENT_DIR = Path.cwd()
DATA_DIR = CURRENT_DIR.parent / 'data'
RAW_DATA_DIR = DATA_DIR / 'raw'
RAW_DATA_FILE = RAW_DATA_DIR / 'complete_pitch_log.csv'

In [9]:
statcast_df = pd.read_csv(RAW_DATA_FILE, low_memory=False)

In [10]:
# Clean to remove unnecessary columns. Only need pitch speed, location, pitch outcome, and other swing metrics
def clean_statcast_df(statcast_df):
        if statcast_df.empty:
                return pd.DataFrame()

        cleaned_sc_df = statcast_df.copy()

        cleaned_sc_df = cleaned_sc_df.dropna(how='all', axis=1) # Drop empty columns
        cleaned_sc_df = cleaned_sc_df.loc[:, ~cleaned_sc_df.columns.str.startswith('fielder')] # Drop columns with no useful info

        cleaned_sc_df["runners_on"] = 0

        for base_col in ["on_1b", "on_2b", "on_3b"]:
                if base_col in cleaned_sc_df.columns:
                        cleaned_sc_df["runners_on"] += pd.to_numeric(cleaned_sc_df[base_col], errors="coerce").fillna(0)


        cleaned_sc_df = cleaned_sc_df.drop(columns=['home_team', 'away_team', 'on_3b', 'on_2b', 'on_1b', 'estimated_ba_using_speedangle', 'estimated_woba_using_speedangle', 'woba_value', 'woba_denom',
                                                'babip_value', 'iso_value', 'home_score', 'away_score', 'bat_score', 'fld_score', 'batter_days_since_prev_game', 'pitcher_days_since_prev_game',
                                                'batter_days_until_next_game', 'pitcher_days_until_next_game', 'n_priorpa_thisgame_player_at_bat', 'n_thruorder_pitcher', 'age_bat', 'age_pit',
                                                'post_away_score', 'post_home_score', 'post_bat_score', 'post_fld_score', 'if_fielding_alignment', 'of_fielding_alignment', 'delta_home_win_exp',
                                                'delta_run_exp', 'estimated_slg_using_speedangle', 'delta_pitcher_run_exp', 'hyper_speed', 'home_score_diff', 'bat_score_diff', 'home_win_exp',
                                                'bat_win_exp', 'age_pit_legacy', 'age_bat_legacy', 'des'])

        return cleaned_sc_df

In [11]:
cleaned = clean_statcast_df(statcast_df)
display(cleaned.head())


,pitch_type,game_date,release_speed,release_pos_x,release_pos_z,player_name,batter,pitcher,events,description,...,api_break_x_batter_in,arm_angle,attack_angle,attack_direction,swing_path_tilt,intercept_ball_minus_batter_pos_x_inches,intercept_ball_minus_batter_pos_y_inches,team,pitch_type_id,runners_on
0,FF,2025-03-27,94.6,-3.58,5.45,"Wells, Austin",669224,642547,NaN,ball,...,-0.07,44.0,NaN,NaN,NaN,NaN,NaN,NYY,5,0
1,FF,2025-03-27,94.2,-3.69,5.42,"Wells, Austin",669224,642547,NaN,ball,...,-0.61,38.7,NaN,NaN,NaN,NaN,NaN,NYY,5,0
2,FF,2025-03-27,93.4,-3.60,5.43,"Wells, Austin",669224,642547,home_run,hit_into_play,...,-0.13,41.6,14.519504,-3.948830,21.480403,31.935837,29.264140,NYY,5,0
3,FF,2025-03-27,95.1,-3.45,5.32,"Judge, Aaron",592450,642547,NaN,ball,...,0.32,44.7,NaN,NaN,NaN,NaN,NaN,NYY,5,0
4,FF,2025-03-27,95.8,-3.48,5.47,"Judge, Aaron",592450,642547,NaN,foul,...,0.57,44.4,6.595282,-1.466255,41.410435,29.692932,25.005637,NYY,5,0


In [12]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import make_pipeline


# -----------------------------------------------------------------------------
# Rolling player-game model
# -----------------------------------------------------------------------------
# This function creates a compact player-game training table from the raw pitch log.
# Each game becomes one row, and each row contains the recent pitch behavior for that
# player plus a custom target representing the offensive value produced during that
# game. The model is intentionally built to avoid leakage by using only information
# from earlier games when predicting any later game.

def build_rolling_player_model(
    cleaned_pitches,
    rolling_games=15,
    min_history_games=3,
    test_fraction=0.2,
):
    """Predict future per-PA offensive value using only prior games in cleaned pitches. The goal
    is a custom offensive-value-per-plate-appearance metric
    """
    if rolling_games < 1:
        raise ValueError("rolling_games must be at least 1")
    if not 0 < test_fraction < 1:
        raise ValueError("test_fraction must be between 0 and 1")

    pitches = cleaned_pitches.copy()
    required_columns = {"game_pk", "game_date", "batter", "description", "events"}
    missing_columns = required_columns.difference(pitches.columns)
    if missing_columns:
        raise ValueError(f"Missing cleaned pitch columns: {sorted(missing_columns)}")

    # Convert raw batter IDs into a numeric player identifier and parse dates to ensure consistent grouping across games
    # and allows sorting for each player's games in chronological order
    pitches["player_id"] = pd.to_numeric(pitches["batter"], errors="coerce")
    pitches["game_date"] = pd.to_datetime(pitches["game_date"], errors="coerce")
    pitches = pitches.dropna(subset=["player_id", "game_pk", "game_date"])
    pitches["player_id"] = pitches["player_id"].astype("int64")
    pitches["description"] = pitches["description"].fillna("").str.lower()
    pitches["events"] = pitches["events"].fillna("").str.lower()

    # Convert raw pitch outcomes into a few interpretable features like swing, contact, chase, whiff, and hard-hit behaviors
    swing_outcomes = {
        "swinging_strike", "swinging_strike_blocked", "foul", "foul_tip",
        "hit_into_play", "foul_bunt", "missed_bunt", "bunt_foul_tip",
    }

    contact_outcomes = {"foul", "foul_tip", "hit_into_play", "foul_bunt", "bunt_foul_tip"}

    pitches["is_swing"] = pitches["description"].isin(swing_outcomes)
    pitches["is_contact"] = pitches["description"].isin(contact_outcomes)
    pitches["is_whiff"] = pitches["description"].isin(
        {"swinging_strike", "swinging_strike_blocked", "missed_bunt"}
    )

    # Use the statcast zone metric to identify pitches outside the strike zone (11-14 are designated outside the zone values)
    pitches["is_outside_zone"] = pd.to_numeric(
        pitches.get("zone", pd.Series(index=pitches.index, dtype="float64")),
        errors="coerce",
    ).between(11, 14)

    # Identify batted balls and hard-hit balls using the launch speed metric. Hard-hit balls are defined as batted balls with a launch speed of 95 mph or higher.
    pitches["is_batted_ball"] = pitches["description"].eq("hit_into_play")
    pitches["launch_speed"] = pd.to_numeric(
        pitches.get("launch_speed", pd.Series(index=pitches.index, dtype="float64")),
        errors="coerce",
    )
    pitches["is_chase_swing"] = pitches["is_swing"] & pitches["is_outside_zone"]
    pitches["is_hard_hit_ball"] = pitches["is_batted_ball"] & pitches["launch_speed"].ge(95)

    # Only keep measurement columns that are relevant to pitch movement, location, and a few other metrics.
    pitch_metrics = [
        "release_speed", "effective_speed", "release_spin_rate", "pfx_x", "pfx_z",
        "plate_x", "plate_z", "bat_speed", "swing_length", "attack_angle",
        "launch_speed", "launch_angle",
    ]
    pitch_metrics = [column for column in pitch_metrics if column in pitches.columns]
    for column in pitch_metrics:
        pitches[column] = pd.to_numeric(pitches[column], errors="coerce")

    # Aggregate pitch-level data into one row per player-game
    group_keys = ["player_id", "game_pk", "game_date"]
    game_features = pitches.groupby(group_keys, as_index=False).agg(
        pitch_count=("description", "size"),
        swing_rate=("is_swing", "mean"),
        swings=("is_swing", "sum"),
        contacts=("is_contact", "sum"),
        whiffs=("is_whiff", "sum"),
        outside_zone_pitches=("is_outside_zone", "sum"),
        chase_swings=("is_chase_swing", "sum"),
        batted_balls=("is_batted_ball", "sum"),
        hard_hit_balls=("is_hard_hit_ball", "sum"),
        **{f"{column}_mean": (column, "mean") for column in pitch_metrics},
    )

    # Rate features are calculated as share-of-eligible-pitches or share-of-swings
    # Substitute missing values when the denominator is zero. Get rid of the original count columns after calculating rates
    game_features["contact_rate"] = game_features["contacts"] / game_features["swings"].replace(0, np.nan)
    game_features["whiff_rate"] = game_features["whiffs"] / game_features["swings"].replace(0, np.nan)
    game_features["chase_rate"] = game_features["chase_swings"] / game_features["outside_zone_pitches"].replace(0, np.nan)
    game_features["hard_hit_rate"] = game_features["hard_hit_balls"] / game_features["batted_balls"].replace(0, np.nan)
    game_features = game_features.drop(
        columns=["swings", "contacts", "whiffs", "outside_zone_pitches", "chase_swings", "batted_balls", "hard_hit_balls"]
    )

    # Create the average offensive value per plate appearance using custom weights for each event type.
    # These weights are intentionally scaled to match the relative ratios used in the game_score notebook:
    # single = 15, double = 30, triple = 45, home run = 60, walk = 15.
    # We keep the same 1:2:3:4 ordering here, while expressing the value on a per-PA scale.
    pa_outcomes = pitches[pitches["events"].ne("")].copy()
    if "at_bat_number" in pa_outcomes.columns:
        pa_outcomes = pa_outcomes.drop_duplicates(["game_pk", "player_id", "at_bat_number"])

    event_value = {
        "single": 1.0,
        "double": 2.0,
        "triple": 3.0,
        "home_run": 4.0,
        "walk": 1.0,
        "intent_walk": 1.0,
        "hit_by_pitch": 1.0,
        "catcher_interf": 1.0,
        "field_error": 1.0,
        "error": 1.0,
    }
    pa_outcomes["offensive_value"] = pa_outcomes["events"].map(event_value).fillna(0.0)
    game_targets = pa_outcomes.groupby(group_keys, as_index=False).agg(
        offensive_value_per_pa=("offensive_value", "mean"),
        plate_appearances=("events", "size"),
    )
    model_data = game_features.merge(game_targets, on=group_keys, how="inner")
    model_data = model_data.sort_values(["player_id", "game_date", "game_pk"]).reset_index(drop=True)


    # Build rolling features by player using only previous games
    # This is the leakage safeguard. Each prediction uses only the batter's earlier
    # games, with the current game's pitch behavior and outcome excluded from the
    # predictors
    rolling_source_columns = [
        column for column in game_features.columns
            if column not in {"player_id", "game_pk", "game_date"}
    ]
    by_player = model_data.groupby("player_id", sort=False)
    model_data["prior_games_available"] = by_player.cumcount()
    feature_columns = ["prior_games_available"]
    for column in rolling_source_columns:
        feature_name = f"{column}_last_{rolling_games}_games"
        model_data[feature_name] = by_player[column].transform(
            lambda values: values.shift(1).rolling(rolling_games, min_periods=1).mean()
        )
        feature_columns.append(feature_name)

    target_column = "offensive_value_per_pa"
    baseline_column = f"prior_performance_mean_last_{rolling_games}_games"
    model_data[baseline_column] = by_player[target_column].transform(
        lambda values: values.shift(1).rolling(rolling_games, min_periods=1).mean()
    )
    feature_columns.append(baseline_column)

    # Require enough prior games to make the rolling baseline meaningful before a
    # row is used in training or testing.
    model_data = model_data[model_data["prior_games_available"] >= min_history_games].copy()

    # Split the training / test data based on chronological date order. The train/test split is
    # meant to portray real-world forecasting so predicting future performance using only past
    # information is what needs to be evaluated
    unique_dates = np.sort(model_data["game_date"].dropna().unique())
    if len(unique_dates) < 2:
        raise ValueError("Need at least two game dates after applying the history requirement")

    cutoff_index = min(max(int(len(unique_dates) * (1 - test_fraction)), 1), len(unique_dates) - 1)
    cutoff_date = unique_dates[cutoff_index]
    train = model_data[model_data["game_date"] < cutoff_date]
    test = model_data[model_data["game_date"] >= cutoff_date]
    if train.empty or test.empty:
        raise ValueError("Chronological split produced an empty train or test set")

    # The feature matrix contains a mix of real-valued pitch metrics and derived rates.
    # Missing values are imputed with the median from the training data, and the random
    # forest then learns nonlinear relationships between recent plate-discipline signals
    # and the target offensiveness measure.
    model = make_pipeline(
        SimpleImputer(strategy="median"),
        RandomForestRegressor(
            n_estimators=300,
            min_samples_leaf=5,
            random_state=42,
            n_jobs=-1,
        ),
    )

    model.fit(train[feature_columns], train[target_column])
    test_predictions = model.predict(test[feature_columns])

    metrics = {
        "rolling_games": rolling_games,
        "min_history_games": min_history_games,
        "train_rows": len(train),
        "test_rows": len(test),
        "test_start_date": pd.Timestamp(cutoff_date),
        "model_mae": mean_absolute_error(test[target_column], test_predictions),
        "baseline_mae": mean_absolute_error(test[target_column], test[baseline_column]),
        "model_rmse": mean_squared_error(test[target_column], test_predictions) ** 0.5,
        "model_r2": r2_score(test[target_column], test_predictions) if len(test) > 1 else np.nan,
    }

    # Store raw predictions alongside the true target and the prior-performance baseline.
    predictions = test[["game_date", "game_pk", "player_id", target_column, baseline_column]].copy()
    predictions["predicted_performance_per_pa"] = test_predictions
    predictions = predictions.rename(columns={baseline_column: "prior_performance_baseline"})

    # Feature importance helps show which inputs matter most to the forest, but it does
    # not prove any causal relationships or superior baseball strategy. Just using to determine which features are
    # most useful for then model
    forest = model.named_steps["randomforestregressor"]
    importance = pd.Series(forest.feature_importances_, index=feature_columns).sort_values(ascending=False)

    return model, metrics, predictions.reset_index(drop=True), importance


ROLLING_GAMES = 15
MIN_HISTORY_GAMES = 3

# Fit the model and inspect the results
rolling_model, model_metrics, test_predictions, feature_importance = build_rolling_player_model(
    cleaned,
    rolling_games=ROLLING_GAMES,
    min_history_games=MIN_HISTORY_GAMES,
)

display(pd.Series(model_metrics, name="value"))
display(test_predictions.head())
display(feature_importance.head(100).rename("importance"))

rolling_games                         15
min_history_games                      3
train_rows                          5010
test_rows                           1361
test_start_date      2026-07-12 00:00:00
model_mae                       0.384442
baseline_mae                    0.377837
model_rmse                      0.505152
model_r2                       -0.063316
Name: value, dtype: object

,game_date,game_pk,player_id,offensive_value_per_pa,prior_performance_baseline,predicted_performance_per_pa
0,2026-07-12,822708,502671,0.50,0.313333,0.528411
1,2026-07-17,823524,502671,0.25,0.330000,0.449623
2,2026-07-19,823521,502671,0.75,0.213333,0.442046
3,2026-07-20,823522,502671,0.25,0.210000,0.455161
4,2026-07-22,823518,502671,0.00,0.226667,0.467054


plate_x_mean_last_15_games              0.060827
prior_performance_mean_last_15_games    0.059136
release_spin_rate_mean_last_15_games    0.058297
attack_angle_mean_last_15_games         0.058265
launch_angle_mean_last_15_games         0.058054
pfx_x_mean_last_15_games                0.056690
bat_speed_mean_last_15_games            0.056684
pfx_z_mean_last_15_games                0.055221
swing_rate_last_15_games                0.055087
pitch_count_last_15_games               0.054226
chase_rate_last_15_games                0.054147
plate_z_mean_last_15_games              0.054040
hard_hit_rate_last_15_games             0.049091
launch_speed_mean_last_15_games         0.047194
swing_length_mean_last_15_games         0.045984
effective_speed_mean_last_15_games      0.043397
prior_games_available                   0.039905
release_speed_mean_last_15_games        0.037057
contact_rate_last_15_games              0.028951
whiff_rate_last_15_games                0.027748
Name: importance, dt

## Overall Interpretation of Results

Results prove that this model is not a great predictor based on the current setup. It doesn't do a good job predicting any better than the baseline. Need to refine much further, potentially make a simpler model that takes into account less features. Need to investigate the features further and see what really is important.